## Step 1: Setup & Hypothesis

**Business question:** Did the tested marketing change (ad exposure) actually move 
conversion, or could the difference be explained by chance?

**H0:** There is no difference in conversion rate between the ad-exposed (treatment) 
group and the PSA (control) group.

**H1:** The ad-exposed group has a different conversion rate than the PSA group.

**Significance level:** α = 0.05

**Dataset:** Marketing A/B Testing (Kaggle) — 588,101 rows, one row per user.
Columns: `test group` (ad/psa), `converted` (bool), `total ads`, `most ads day`, `most ads hour`.

In [1]:
import pandas as pd

df = pd.read_csv('../data/marketing_AB.csv')
df.head()
df.info()
df.describe(include='all')

<class 'pandas.DataFrame'>
RangeIndex: 588101 entries, 0 to 588100
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   Unnamed: 0     588101 non-null  int64
 1   user id        588101 non-null  int64
 2   test group     588101 non-null  str  
 3   converted      588101 non-null  bool 
 4   total ads      588101 non-null  int64
 5   most ads day   588101 non-null  str  
 6   most ads hour  588101 non-null  int64
dtypes: bool(1), int64(4), str(2)
memory usage: 27.5 MB


,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
count,588101.000000,5.881010e+05,588101,588101,588101.000000,588101,588101.000000
unique,NaN,NaN,2,2,NaN,7,NaN
top,NaN,NaN,ad,False,NaN,Friday,NaN
freq,NaN,NaN,564577,573258,NaN,92608,NaN
mean,294050.000000,1.310692e+06,NaN,NaN,24.820876,NaN,14.469061
std,169770.279668,2.022260e+05,NaN,NaN,43.715181,NaN,4.834634
min,0.000000,9.000000e+05,NaN,NaN,1.000000,NaN,0.000000
25%,147025.000000,1.143190e+06,NaN,NaN,4.000000,NaN,11.000000
50%,294050.000000,1.313725e+06,NaN,NaN,13.000000,NaN,14.000000
75%,441075.000000,1.484088e+06,NaN,NaN,27.000000,NaN,18.000000


## Step 2: Sanity Checks — Group Sizes & Randomization

Before testing anything, confirm the data is clean and understand the group split.

**Findings:**
- Duplicate user ids: 0 — no user appears more than once, safe to treat rows as independent.
- Group sizes: `ad` = 564,577 (96%), `psa` = 23,524 (4%).
- The split is heavily imbalanced (not 50/50). This is common in real-world ad tests — 
  companies often hold out a small control rather than withholding ads from half of traffic.
  It's not a randomization bug, but it reduces statistical power on the control side, 
  which we check formally in Step 5.

In [2]:
# Drop the junk index column
df = df.drop(columns=['Unnamed: 0'])

# 1. Check for duplicate users (should be 0 if one row per user)
print("Duplicate user ids:", df['user id'].duplicated().sum())

# 2. Group sizes — sample ratio check
group_counts = df['test group'].value_counts()
print(group_counts)
print(group_counts / len(df))

Duplicate user ids: 0
test group
ad     564577
psa     23524
Name: count, dtype: int64
test group
ad     0.96
psa    0.04
Name: count, dtype: float64


## Step 3: Conversion Rates, Lift & Two-Proportion Z-Test

**Why a z-test and not a t-test:** conversion is a binary outcome, and we're comparing 
two proportions — not two means of continuous data. A t-test assumes continuous, 
roughly-normal data and is the wrong tool here, even if it happens to run without error.

**Results:**
| Group | Conversions | n | Conversion Rate |
|---|---|---|---|
| ad (treatment) | 14,423 | 564,577 | 2.55% |
| psa (control) | 420 | 23,524 | 1.79% |

- Absolute lift: **+0.77 percentage points**
- Relative lift: **+43.09%**
- Z-statistic: **7.37**
- P-value: **≈ 0.0000000000** (effectively zero)

The difference is statistically significant — the ad group converts at a meaningfully 
higher rate than control. But p-value alone doesn't tell us the size of the effect, 
which is why we compute a confidence interval next.

In [3]:
# Conversion rate per group
conv_summary = df.groupby('test group')['converted'].agg(['sum', 'count', 'mean'])
conv_summary.columns = ['conversions', 'n', 'conversion_rate']
print(conv_summary)

# Absolute and relative lift (ad vs. psa)
p_ad = conv_summary.loc['ad', 'conversion_rate']
p_psa = conv_summary.loc['psa', 'conversion_rate']

abs_lift = p_ad - p_psa
rel_lift = abs_lift / p_psa

print(f"\nControl (psa) conversion rate: {p_psa:.4%}")
print(f"Treatment (ad) conversion rate: {p_ad:.4%}")
print(f"Absolute lift: {abs_lift:.4%}")
print(f"Relative lift: {rel_lift:.2%}")

            conversions       n  conversion_rate
test group                                      
ad                14423  564577         0.025547
psa                 420   23524         0.017854

Control (psa) conversion rate: 1.7854%
Treatment (ad) conversion rate: 2.5547%
Absolute lift: 0.7692%
Relative lift: 43.09%


In [4]:
from statsmodels.stats.proportion import proportions_ztest

counts = conv_summary.loc[['ad', 'psa'], 'conversions'].values
nobs = conv_summary.loc[['ad', 'psa'], 'n'].values

z_stat, p_value = proportions_ztest(count=counts, nobs=nobs, alternative='two-sided')

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.10f}")

Z-statistic: 7.3701
P-value: 0.0000000000


## Step 4: 95% Confidence Interval for the Lift

The p-value tells us a difference likely exists; the CI tells us the plausible 
*range* of that difference — which is what actually justifies a business decision.

**Results:**
- Absolute lift: 0.77%, 95% CI: **[0.60%, 0.94%]**
- Relative lift: 43.09%, 95% CI: **[33.33%, 52.84%]**

The interval is tight and entirely positive — even at the low end, the ad group 
converts at least 33% better than control. This is a strong basis for a "ship it" 
recommendation, not just a low p-value.

In [5]:
import numpy as np
from scipy.stats import norm

# Standard error of the difference in proportions
se_diff = np.sqrt(
    (p_ad * (1 - p_ad)) / conv_summary.loc['ad', 'n'] +
    (p_psa * (1 - p_psa)) / conv_summary.loc['psa', 'n']
)

# 95% CI for the absolute lift
z_critical = norm.ppf(0.975)  # 1.96
ci_lower = abs_lift - z_critical * se_diff
ci_upper = abs_lift + z_critical * se_diff

print(f"Absolute lift: {abs_lift:.4%}")
print(f"95% CI: [{ci_lower:.4%}, {ci_upper:.4%}]")

# Same thing, expressed as relative lift for easier business interpretation
rel_ci_lower = ci_lower / p_psa
rel_ci_upper = ci_upper / p_psa
print(f"\nRelative lift: {rel_lift:.2%}")
print(f"95% CI (relative): [{rel_ci_lower:.2%}, {rel_ci_upper:.2%}]")

Absolute lift: 0.7692%
95% CI: [0.5951%, 0.9434%]

Relative lift: 43.09%
95% CI (relative): [33.33%, 52.84%]


## Step 5: Retrospective Power / Sample-Size Check

Given the sample sizes we actually had, could we reliably detect an effect of this size?

**Results:**
- Effect size (Cohen's h): **0.053** (small-to-moderate)
- Achieved power: **100%**
- Minimum control (psa) sample size needed for 80% power: **~2,910**
- Actual control (psa) sample size: **23,524**

The test was massively over-powered — we had ~8x more control users than needed. 
Useful for future test design: a control group this large wasn't necessary to detect 
an effect of this size.

In [6]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# Effect size (Cohen's h) based on the observed proportions
effect_size = proportion_effectsize(p_ad, p_psa)
print(f"Effect size (Cohen's h): {effect_size:.4f}")

# Power analysis setup
power_analysis = NormalIndPower()

# 1. What power did we actually have, given our real sample sizes?
n_ad = conv_summary.loc['ad', 'n']
n_psa = conv_summary.loc['psa', 'n']
ratio = n_ad / n_psa  # unequal group sizes

achieved_power = power_analysis.power(
    effect_size=effect_size,
    nobs1=n_psa,          # power calc conventionally anchors on the smaller/control group
    alpha=0.05,
    ratio=ratio
)
print(f"Achieved power: {achieved_power:.4%}")

# 2. What's the minimum control-group sample size we'd have NEEDED
#    for 80% power to detect this effect size?
required_n_psa = power_analysis.solve_power(
    effect_size=effect_size,
    alpha=0.05,
    power=0.80,
    ratio=ratio
)
print(f"Minimum required psa (control) sample size for 80% power: {required_n_psa:.0f}")
print(f"Actual psa (control) sample size: {n_psa}")

Effect size (Cohen's h): 0.0530
Achieved power: 100.0000%
Minimum required psa (control) sample size for 80% power: 2910
Actual psa (control) sample size: 23524


## Step 6: Segmentation — Does the Effect Hold Across Subgroups?

Checking whether the overall lift is consistent across day-of-week and time-of-day, 
or driven by one subgroup (Simpson's paradox risk).

### By day of week
- Every day shows a **positive** lift (ad > psa) — no reversal, no Simpson's paradox.
- Magnitude varies: Tuesday shows the strongest effect (+111% relative lift, 
  p≈0.0000007); Sunday (+20%, p=0.16) and Thursday (+7%, p=0.55) are not statistically 
  significant at this sample size — likely underpowered on those days rather than 
  evidence the ad doesn't work.

### By time of day
- All four time buckets (Morning, Afternoon, Evening, Night) show a **positive and 
  statistically significant** lift (all p < 0.01) — the strongest and most consistent 
  cut in this analysis.
- Morning (+69%) and Night (+66%) show the largest relative lift; Afternoon (+37%) and 
  Evening (+29%) are more moderate but still clearly significant.

### Overall read on Step 6
The treatment effect is **directionally consistent across every day and every time 
bucket** — no subgroup shows a reversal, so there's no evidence of Simpson's paradox. 
The effect is strongest on Tuesday and during Morning/Night hours, and weaker (though 
still real) on Sunday/Thursday and during Afternoon/Evening. This supports treating 
the overall +43% lift as a genuine, broad-based effect rather than one driven by a 
single subgroup.

In [7]:
def segment_conversion_test(df, segment_col):
    results = []
    for segment_value in df[segment_col].unique():
        sub = df[df[segment_col] == segment_value]
        sub_summary = sub.groupby('test group')['converted'].agg(['sum', 'count', 'mean'])

        # Skip if either group is missing or too small in this slice
        if 'ad' not in sub_summary.index or 'psa' not in sub_summary.index:
            continue

        counts = sub_summary.loc[['ad', 'psa'], 'sum'].values
        nobs = sub_summary.loc[['ad', 'psa'], 'count'].values

        if min(nobs) < 30:  # too small to trust a z-test here
            continue

        z, p = proportions_ztest(count=counts, nobs=nobs, alternative='two-sided')

        p_ad_seg = sub_summary.loc['ad', 'mean']
        p_psa_seg = sub_summary.loc['psa', 'mean']

        results.append({
            'segment': segment_value,
            'n_ad': nobs[0],
            'n_psa': nobs[1],
            'conv_ad': p_ad_seg,
            'conv_psa': p_psa_seg,
            'rel_lift': (p_ad_seg - p_psa_seg) / p_psa_seg if p_psa_seg > 0 else np.nan,
            'p_value': p
        })

    return pd.DataFrame(results).sort_values('segment').reset_index(drop=True)

day_results = segment_conversion_test(df, 'most ads day')
print(day_results)

     segment   n_ad  n_psa   conv_ad  conv_psa  rel_lift       p_value
0     Friday  88805   3803  0.022465  0.016303  0.377971  1.156888e-02
1     Monday  83571   3502  0.033241  0.022559  0.473553  5.078178e-04
2   Saturday  78802   2858  0.021307  0.013996  0.522354  7.483818e-03
3     Sunday  82332   3059  0.024620  0.020595  0.195430  1.571861e-01
4   Thursday  79077   3905  0.021637  0.020230  0.069532  5.547506e-01
5    Tuesday  74572   2907  0.030440  0.014448  1.106909  6.634401e-07
6  Wednesday  77418   3490  0.025356  0.015759  0.608945  3.764133e-04


In [9]:
def bucket_hour(h):
    if 5 <= h < 12:
        return 'Morning (5-11)'
    elif 12 <= h < 17:
        return 'Afternoon (12-16)'
    elif 17 <= h < 21:
        return 'Evening (17-20)'
    else:
        return 'Night (21-4)'

df['time_of_day'] = df['most ads hour'].apply(bucket_hour)

hour_results = segment_conversion_test(df, 'time_of_day')
print(hour_results)

             segment    n_ad  n_psa   conv_ad  conv_psa  rel_lift   p_value
0  Afternoon (12-16)  213320   9531  0.027531  0.020145  0.366676  0.000014
1    Evening (17-20)  121672   4914  0.028248  0.021978  0.285287  0.009006
2     Morning (5-11)  137268   5750  0.021163  0.012522  0.690099  0.000007
3       Night (21-4)   92317   3329  0.023918  0.014419  0.658784  0.000388


## Step 7: Executive Memo — Ship / No-Ship Recommendation

**Decision: Ship it.**

The ad campaign drove a statistically significant and practically meaningful increase 
in conversion rate. Users exposed to the ad converted at 2.55%, versus 1.79% for the 
PSA control group — a lift of +0.77 percentage points, or +43% relative.

**Confidence:** Very high. The 95% confidence interval for the relative lift is 
[33%, 53%] — even in the most conservative case, the ad still meaningfully 
outperforms control. The test was also over-powered (100% achieved power vs. an 
80% target), so this isn't a fragile, sample-starved result.

**Robustness:** The effect holds directionally across every day of the week and every 
time-of-day bucket tested — no subgroup shows a reversal. It's strongest on Tuesdays 
and during morning/night hours, and more modest (but still positive) on Sundays, 
Thursdays, and afternoons/evenings.

**Risk if wrong:** Low. Given the size and consistency of the effect, the main risk 
isn't that the ad doesn't work — it's overstating the lift if future traffic mix 
shifts toward the weaker-performing segments (Sunday/Thursday, afternoon/evening). 
Recommend monitoring conversion rate by day/time post-launch to confirm the pattern holds.

**What to test next:**
1. Test ad *creative variants* against each other now that "ad vs. no ad" is settled.
2. Since the control group was ~8x larger than statistically necessary (2,910 vs. 
   23,524), future tests in this program can run with a smaller control allocation, 
   freeing up more users to see the ad sooner.
3. Investigate *why* morning/night and Tuesday show stronger lift — could inform 
   ad scheduling/budget allocation rather than a blanket always-on strategy.